# Demo (Lensless Computational Imaging)

### Конфигурация
Для своих данных надо поменять `DATASET_ZIP_URL`. Архив должен содержать директории `lensless/`, `masks/` и `lensed/`.

In [ ]:
REPO_ID = "ArTiK13/Lensless-Computational-Imaging"
DATASET_ZIP_URL = "https://drive.google.com/uc?export=download&id=1aw2FMI4sstuRmRIqure4UhX685uIYWty" # сюда надо ссылку на файл

ZIP_PATH = "content/lensless_dataset.zip"
DATA_DIR = "content/lensless_dataset"

MODEL_NAME = "modular_post_only"
SAVE_NAME = "demo_reconstructions"
DEVICE = "cuda"

### Скачивание репозитория, установка библиотек и чекпоинтов

In [ ]:
repo_url = f"https://github.com/{REPO_ID}.git"
!git clone $repo_url
repo_name = REPO_ID.split("/")[1]
%cd $repo_name

In [ ]:
!pip install -q -r requirements_demo.txt

In [ ]:
!python3 scripts/download_checkpoints.py --output-dir checkpoints

### Скачивание данных

In [ ]:
from pathlib import Path
import shutil
import gdown

zip_path = Path(ZIP_PATH)
data_dir = Path(DATA_DIR)
data_dir.mkdir(parents=True, exist_ok=True)

gdown.download(url=DATASET_ZIP_URL, output=str(zip_path), quiet=False)
shutil.unpack_archive(str(zip_path), str(data_dir))

### Инференс и показ результатов

In [ ]:
checkpoint_by_model = {
    "leadmm20": "checkpoints/leadmm20.pth",
    "modular_pre_post": "checkpoints/modular_pre_post.pth",
    "modular_pre_only": "checkpoints/modular_pre_only.pth",
    "modular_post_only": "checkpoints/modular_post_only.pth",
}

checkpoint_path = checkpoint_by_model.get(MODEL_NAME)
load_args = "inferencer.skip_model_load=true" if checkpoint_path is None else f"inferencer.from_pretrained={checkpoint_path} inferencer.skip_model_load=false"

!python3 inference.py -cn inference_custom datasets.test.root_dir=$data_dir inferencer.save_path=$SAVE_NAME inferencer.device=$DEVICE inferencer.compute_metrics=false model=$MODEL_NAME $load_args

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

recon_dir = Path("data/saved") / SAVE_NAME / "test"
recon_path = sorted(recon_dir.glob("*.png"))[0]
image_id = recon_path.stem

items = [
    ("измерение", data_dir / "lensless" / f"{image_id}.png"),
    ("реконструкция", recon_path),
    ("таргет", data_dir / "lensed" / f"{image_id}.png"),
]

plt.figure(figsize=(5 * len(items), 5))
for i, (title, path) in enumerate(items, 1):
    plt.subplot(1, len(items), i)
    plt.imshow(Image.open(path).convert("RGB"))
    plt.title(title)
    plt.axis("off")
plt.show()

In [ ]:
gt_dir = data_dir / "lensed"
!python3 scripts/calculate_metrics.py --gt-dir $gt_dir --recon-dir $recon_dir